# DuckDB

In [17]:
import duckdb
import pandas as pd

## Visão geral

In [18]:
# Columns
path_parquet = 'Genius-song-lyrics-cleaned/data/*.parquet'
relation = duckdb.read_parquet(path_parquet)

print("## COLUMNS ##")
for i, column in enumerate(relation.columns, start=1):
    print(f"{i}. {column}")

## COLUMNS ##
1. title
2. tag
3. artist
4. year
5. views
6. features
7. lyrics
8. id
9. language_cld3
10. language_ft
11. language
12. lyrics_clean
13. char_len
14. word_len


In [19]:
# 20 first rows
query = f"""
    SELECT * FROM read_parquet('{path_parquet}') LIMIT 20;
"""

duckdb.sql(query).show()

┌────────────────────────────┬─────────┬───────────┬───────┬────────┬──────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [20]:
# Most relevant columns
query = f"""
    SELECT
        id,
        title,
        tag,
        artist,
        features,
        year,
        views,
        word_len,
        language_cld3,
        language_ft,
        language
    FROM read_parquet('{path_parquet}') LIMIT 20;
"""

duckdb.sql(query).show()

┌───────┬────────────────────────────┬─────────┬───────────┬──────────────────────────────────────────────┬───────┬────────┬──────────┬───────────────┬─────────────┬──────────┐
│  id   │           title            │   tag   │  artist   │                   features                   │ year  │ views  │ word_len │ language_cld3 │ language_ft │ language │
│ int64 │          varchar           │ varchar │  varchar  │                   varchar                    │ int64 │ int64  │  int64   │    varchar    │   varchar   │ varchar  │
├───────┼────────────────────────────┼─────────┼───────────┼──────────────────────────────────────────────┼───────┼────────┼──────────┼───────────────┼─────────────┼──────────┤
│     1 │ Killa Cam                  │ rap     │ Cam'ron   │ {"Cam\\'ron","Opera Steve"}                  │  2004 │ 173166 │      444 │ en            │ en          │ en       │
│     3 │ Can I Live                 │ rap     │ JAY-Z     │ {}                                           │  1996 │

## Unique values

In [21]:
# Count of unique values (distinct)
query = f"""
    SELECT
        approx_count_distinct(artist) AS distinct_artists,
        approx_count_distinct(tag) AS distinct_tags,
        approx_count_distinct(title) AS distinct_song_titles,
        approx_count_distinct(year) AS distinct_years
    FROM read_parquet('{path_parquet}')
    WHERE language='en';
"""

duckdb.sql(query).show()

┌──────────────────┬───────────────┬──────────────────────┬────────────────┐
│ distinct_artists │ distinct_tags │ distinct_song_titles │ distinct_years │
│      int64       │     int64     │        int64         │     int64      │
├──────────────────┼───────────────┼──────────────────────┼────────────────┤
│           372197 │             6 │              1893454 │            636 │
└──────────────────┴───────────────┴──────────────────────┴────────────────┘



In [22]:
# Average word count of songs
query = f"SELECT avg(word_len) FROM read_parquet('{path_parquet}');"

duckdb.sql(query).show()

┌────────────────────┐
│   avg(word_len)    │
│       double       │
├────────────────────┤
│ 234.28213254665758 │
└────────────────────┘



In [23]:
# Unique tags (genres)
query = f"""
    SELECT DISTINCT "tag" FROM read_parquet('{path_parquet}');
"""

duckdb.sql(query).show()

┌─────────┐
│   tag   │
│ varchar │
├─────────┤
│ rock    │
│ pop     │
│ misc    │
│ rap     │
│ rb      │
│ country │
└─────────┘



## Language

In [24]:
# Checking the language, language_ft (fasttext) and language_cld3 (google compact language detector v3) columns
query = f"""
    SELECT
        COUNT(*) FILTER (language IS DISTINCT FROM language_ft)
            AS lang_vs_lang_ft,

        COUNT(*) FILTER (language IS DISTINCT FROM language_ft)
            AS lang_vs_lang_cld3,

        COUNT(*) FILTER (language IS DISTINCT FROM language_ft)
            AS lang_ft_vs_lang_cld3

        FROM read_parquet('{path_parquet}')
        WHERE language='en';
"""

duckdb.sql(query).show()

┌─────────────────┬───────────────────┬──────────────────────┐
│ lang_vs_lang_ft │ lang_vs_lang_cld3 │ lang_ft_vs_lang_cld3 │
│      int64      │       int64       │        int64         │
├─────────────────┼───────────────────┼──────────────────────┤
│               0 │                 0 │                    0 │
└─────────────────┴───────────────────┴──────────────────────┘



## Artist

In [25]:
# General statistics about artist
query = f"""
    SELECT
        artist,
        COUNT(*) AS total_song_count
    FROM read_parquet('{path_parquet}')
    WHERE language='en'
    GROUP BY artist
    ORDER BY total_song_count DESC
    LIMIT 50;
"""

duckdb.sql(query).show(max_rows=50)

┌─────────────────────────────┬──────────────────┐
│           artist            │ total_song_count │
│           varchar           │      int64       │
├─────────────────────────────┼──────────────────┤
│ Genius English Translations │            13671 │
│ The Grateful Dead           │             2110 │
│ Emily Dickinson             │             1619 │
│ Abraham Lincoln             │             1492 │
│ Lil B                       │             1310 │
│ Frank Zappa                 │             1281 │
│ William Shakespeare         │             1258 │
│ Gucci Mane                  │             1245 │
│ Lil Wayne                   │             1213 │
│ KIDZ BOP Kids               │             1199 │
│ Guided by Voices            │             1188 │
│ Holy Bible (KJV)            │             1187 │
│ Tendon Levey                │             1183 │
│ Juice WRLD                  │             1170 │
│ Charles Dickens             │             1092 │
│ Frank Sinatra               │

In [26]:
# These are some artists that should be deleted
query = f"""
    SELECT
        COUNT(*),
        artist
    FROM read_parquet('{path_parquet}')
    WHERE
        language='en'
        AND artist ILIKE '%genius%'
        AND artist ILIKE '%translation%'
    GROUP BY artist
    ORDER BY artist ASC;
"""

duckdb.sql(query).show()

┌──────────────┬──────────────────────────────────┐
│ count_star() │              artist              │
│    int64     │             varchar              │
├──────────────┼──────────────────────────────────┤
│           34 │ Genius  (Chinese Translations)   │
│            4 │ Genius  (Japanese Translations)  │
│           12 │ Genius Arabic Translations ( )   │
│        13671 │ Genius English Translations      │
│            2 │ Genius Farsi Translations ( )    │
│            5 │ Genius Russian Translations ( )  │
│           36 │ Genius South Africa Translations │
│            4 │ Genius Swedish Translations      │
│           45 │ Genius Translations              │
└──────────────┴──────────────────────────────────┘



In [27]:
# Ordering artist by views ascending (least popular first)
# just to see if there are any other anomalies
# query = f"""
#     SELECT
#         artist,
#         views
#     FROM read_parquet('{path_parquet}')
#     WHERE language='en'
#     GROUP BY artist, views
#     ORDER BY views ASC;
# """

# duckdb.sql(query).show(max_rows=100)

In [28]:
# Checking cases with 'Lyrics' or 'Song' in artist name
query = f"""
    SELECT
        artist,
    FROM read_parquet('{path_parquet}')
    WHERE language='en'
    AND (artist ILIKE '%lyric%'
        OR artist ILIKE '%song%')
    GROUP BY artist
"""

duckdb.sql(query).show(max_rows=30)

┌────────────────────────────────────────────────────┐
│                       artist                       │
│                      varchar                       │
├────────────────────────────────────────────────────┤
│ Trey Songz                                         │
│ LyricalSavage                                      │
│ Lyriciss                                           │
│ Children Songs                                     │
│ Ibsmiah songs                                      │
│ Lyrics Born                                        │
│ Lyricks Fresh                                      │
│ Stamo Songz                                        │
│ Songa                                              │
│ Hillsong Kids                                      │
│ No Words (Skit) Lyrics - Hopsin                    │
│ Ocean Drive Lyrics - Duke Dumont                   │
│ Paradise Lyrics - Logic                            │
│ 50 Best Songs of 2015 Lyrics - Genius              │
│ Criss Ly

## NULL cases

In [29]:
# Checking NULL cases
query = f"""
    SELECT
        COUNT(*) FILTER (WHERE id IS NULL)              AS null_ids,
        COUNT(*) FILTER (WHERE title IS NULL)           AS null_titles,
        COUNT(*) FILTER (WHERE tag IS NULL)             AS null_tag,
        COUNT(*) FILTER (WHERE artist IS NULL)          AS null_artists,
        COUNT(*) FILTER (WHERE year IS NULL)            AS null_year,
        COUNT(*) FILTER (WHERE views IS NULL)           AS null_views,
        COUNT(*) FILTER (WHERE features IS NULL)        AS null_features,
        COUNT(*) FILTER (WHERE lyrics IS NULL)          AS null_lyrics,
        COUNT(*) FILTER (WHERE language_cld3 IS NULL)   AS null_lang_cld3,
        COUNT(*) FILTER (WHERE language_ft IS NULL)     AS null_lang_ft,
        COUNT(*) FILTER (WHERE language IS NULL)        AS null_lang,
        COUNT(*) FILTER (WHERE lyrics_clean IS NULL)    AS null_lyrics_clean,
        COUNT(*) FILTER (WHERE char_len IS NULL)        AS null_char_len,
        COUNT(*) FILTER (WHERE word_len IS NULL)        AS null_word_len
    FROM read_parquet('{path_parquet}')
    WHERE language='en';
"""

duckdb.sql(query).show()

┌──────────┬─────────────┬──────────┬──────────────┬───────────┬────────────┬───────────────┬─────────────┬────────────────┬──────────────┬───────────┬───────────────────┬───────────────┬───────────────┐
│ null_ids │ null_titles │ null_tag │ null_artists │ null_year │ null_views │ null_features │ null_lyrics │ null_lang_cld3 │ null_lang_ft │ null_lang │ null_lyrics_clean │ null_char_len │ null_word_len │
│  int64   │    int64    │  int64   │    int64     │   int64   │   int64    │     int64     │    int64    │     int64      │    int64     │   int64   │       int64       │     int64     │     int64     │
├──────────┼─────────────┼──────────┼──────────────┼───────────┼────────────┼───────────────┼─────────────┼────────────────┼──────────────┼───────────┼───────────────────┼───────────────┼───────────────┤
│        0 │         127 │        0 │            0 │         0 │          0 │             0 │           0 │              0 │            0 │         0 │                 0 │             

In [30]:
# Taking a look at the lyrics of null titles
# query = f"""
#     SELECT
#         artist,
#         lyrics
#     FROM read_parquet('{path_parquet}')
#     WHERE language='en' AND title IS NULL;
# """

# df_null_titles = duckdb.sql(query).df()

# print(f"Total rows found: {len(df_null_titles)}\n")

# if not df_null_titles.empty:
#     for idx, row in df_null_titles.head(10).iterrows():
#         print(f"Song #{idx + 1} | Artist: {row['artist']}")
#         print("-" * 40)

#         # Split lyrics by newline and grab the first 5 lines
#         lyric_lines = str(row['lyrics']).split('\n')
#         five_lines_preview = "\n".join(lyric_lines[:5])

#         print(five_lines_preview)

#         # Add a trailing indicator if the lyrics continue
#         if len(lyric_lines) > 5:
#             print("...")
# else:
#     print("No rows with NULL titles found!")

In [31]:
# Songs that have NULL titles are mostly from unpopular artists,
# so it might be better to just get rid of them

query = f"""
    SELECT
        artist
    FROM read_parquet('{path_parquet}')
    WHERE
        language='en' AND
        title IS NULL
    ORDER BY views DESC;
"""

duckdb.sql(query).show()
# To view all found:
# duckdb.sql(query).show(max_rows=127)

┌──────────────────────┐
│        artist        │
│       varchar        │
├──────────────────────┤
│ Tally Hall           │
│ Trippie Redd         │
│ Touch Amor           │
│ Pinegrove            │
│ RATKING              │
│ Elevation Worship    │
│ OSHUN                │
│ yxngxr1              │
│ Afourteen            │
│ VELVETEARS           │
│     ·                │
│     ·                │
│     ·                │
│ Elliot James Mulhern │
│ Slim Gutz            │
│ Lawfermz             │
│ X Anonymous          │
│ KittyWilson          │
│ Azia (NYC)           │
│ Cloudwatch           │
│ Mathilda             │
│ StonedAKhana         │
│ Grafhic              │
└──────────────────────┘
  127 rows (20 shown) 



## Duplicates

In [32]:
# Total of EN songs vs total of unique EN songs
# query = f"""
#     SELECT
#         COUNT(*) AS total_song_titles,
#         approx_count_distinct(title) AS distinct_song_titles,
#         (total_song_titles - distinct_song_titles) AS 'duplicate_song_titles'
#     FROM read_parquet('{path_parquet}')
#     WHERE language='en';
# """

# duckdb.sql(query).show()

In [33]:
# CHECK: duplicate freq is showing 0.5, which doesn't make much sense
# Checking same song titles where lyrics are different
# (using lyrics_clean to account for minor differences in tags)
# print("Checking duplicate songs where title and lyrics are the same:")
# query_metrics = f"""
#     WITH group_count AS (
#         SELECT
#             title,
#             md5(lyrics_clean) AS lyrics_hash,
#             COUNT(*) AS cases
#         FROM read_parquet('{path_parquet}')
#         WHERE language='en'
#             AND lyrics_clean IS NOT NULL
#         GROUP BY title, md5(lyrics_clean)
#     )

#     SELECT
#         SUM(cases) AS total_rows,
#         COUNT(*) AS unique_combinations,
#         (ANY_VALUE(total_songs) - COUNT(*)) AS total_duplicate_rows,
#         ROUND(((ANY_VALUE(total_songs) - COUNT(*)) / SUM(cases)) * 100, 2) AS duplicate_freq,
#         SUM(CASE WHEN cases > 1 THEN 1 ELSE 0 END) AS total_duplicate_songs
#     FROM group_count,
#         (SELECT COUNT(*) AS total_songs 
#         FROM read_parquet('{path_parquet}') 
#         WHERE language = 'en');
# """

# duckdb.sql(query_metrics).show()

# # Showing the most common duplicates
# query_sample = f"""
#     SELECT 
#         title, 
#         COUNT(*) AS occurrence_count
#     FROM read_parquet('{path_parquet}')
#     WHERE language = 'en'
#         AND lyrics_clean IS NOT NULL
#     GROUP BY title, md5(lyrics_clean)
#     HAVING occurrence_count > 1
#     ORDER BY occurrence_count DESC
#     LIMIT 15;
# """

# duckdb.sql(query_sample).show()

# # These are duplicates where their lyrics are the same, but have different artists covering them
# print("""
# Duplicate songs are all covers by different artists.
# Checking cases where title, artist and lyrics are the same:
# """)
# query_sample = f"""
#     SELECT 
#         title,
#         COUNT(*) AS occurrence_count
#     FROM read_parquet('{path_parquet}')
#     WHERE language = 'en'
#         AND lyrics_clean IS NOT NULL
#     GROUP BY title, artist, md5(lyrics_clean)
#     HAVING occurrence_count > 1
#     ORDER BY occurrence_count DESC
#     LIMIT 15;
# """

# duckdb.sql(query_sample).show()

## Instrumental songs

In [41]:
# Checking if there are any instrumental songs we could delete
query = f"""
    SELECT
        COUNT(*) AS instrumental_count,
        lyrics
    FROM read_parquet('{path_parquet}')
    WHERE language='en'
    AND (lyrics ILIKE '%instrumental%'
        OR lyrics ILIKE '%tba%'
        OR lyrics ILIKE '%no lyrics%')
    AND length(lyrics) < 50
    GROUP BY lyrics;
"""

duckdb.sql(query).show()

┌────────────────────┬─────────┐
│ instrumental_count │ lyrics  │
│       int64        │ varchar │
└────────────────────┴─────────┘
             0 rows           



## Extreme length outliers (corrupted text)

In [43]:
query = f"""
    SELECT
        MIN(length(lyrics_clean)) AS min_chars,
        MAX(length(lyrics_clean)) AS max_chars,
        AVG(length(lyrics_clean)) AS avg_chars
    FROM read_parquet('{path_parquet}')
    WHERE language='en';
"""

duckdb.sql(query).show()

┌───────────┬───────────┬────────────────────┐
│ min_chars │ max_chars │     avg_chars      │
│   int64   │   int64   │       double       │
├───────────┼───────────┼────────────────────┤
│         0 │     99262 │ 1251.0198903561677 │
└───────────┴───────────┴────────────────────┘



In [68]:
# Empty lyrics
query = f"""
    SELECT
        COUNT(*)
    FROM read_parquet('{path_parquet}')
    WHERE language = 'en'
    AND char_len = 0
"""

duckdb.sql(query).show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          117 │
└──────────────┘



In [69]:
# 1372 lyrics with less than 50 characters
query = f"""
    WITH group_count AS (
        SELECT
            COUNT(*)
        FROM read_parquet('{path_parquet}')
        WHERE language = 'en'
        AND char_len < 50
        GROUP BY md5(lyrics_clean)
    )

    SELECT COUNT(*) FROM group_count;
"""

duckdb.sql(query).show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1372 │
└──────────────┘



In [70]:
# Longest characters outlier

query = f"""
    SELECT
        artist,
        title,
        lyrics_clean
    FROM read_parquet('{path_parquet}')
    WHERE language = 'en'
    AND char_len = 99262;
"""

duckdb.sql(query).show()

┌────────────────┬─────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
# TODO: lyrics that are too long, poetry, books

## Poetry

In [35]:
assert False

AssertionError: 

In [ ]:
# Sample de 10000 aleatórios
query = f"""
    SELECT * FROM read_parquet('{path_parquet}') WHERE language='en' USING SAMPLE 10000; 
"""

df_sample = duckdb.query(query).df()

In [ ]:
df_sample.head(20)

# Polars

In [ ]:
# import polars as pl

# # lazyframe do polars com todos os .parquet
# lf = pl.scan_parquet("./Genius-song-lyrics-cleaned/data/train-*.parquet")

# query = (
#     lf
#     .group_by("tag")
#     .agg(pl.len().alias("total_musicas"))
#     .sort("total_musicas", descending=True)
#     .head(10)
# )

# resultado = query.collect()

# print(resultado)

In [ ]:
# print(type(lf))

In [ ]:
# for col in lf.columns:
#     print(f"{col}: {lf[col].nunique()} valores únicos")